In [4]:
import os
import getpass

os.environ["TOGETHER_API_KEY"] = getpass.getpass("TOGETHER_API_KEY")

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)

loan_knowledge_resources = directory_loader.load()

In [6]:
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter


def tiktoken_len(text):
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(
        text,
    )
    return len(tokens)


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=0,
    length_function=tiktoken_len,
)

loan_knowledge_chunks = text_splitter.split_documents(loan_knowledge_resources)

In [7]:
from langchain_community.vectorstores import Qdrant
from langchain_together import TogetherEmbeddings
import os

embedding_model = TogetherEmbeddings(
    model="BAAI/bge-large-en-v1.5",
    together_api_key=os.environ["TOGETHER_API_KEY"],
)
qdrant_vectorstore = Qdrant.from_documents(
    documents=loan_knowledge_chunks, embedding=embedding_model, location=":memory:"
)
qdrant_retriever = qdrant_vectorstore.as_retriever()

In [8]:
from langchain_core.prompts import ChatPromptTemplate

HUMAN_TEMPLATE = """
#CONTEXT:
{context}

QUERY:
{query}

Use the provide context to answer the provided user query. Only use the provided context to answer the query. If you do not know the answer, or it's not contained in the provided context respond with "I don't know"
"""

chat_prompt = ChatPromptTemplate.from_messages([("human", HUMAN_TEMPLATE)])

In [9]:
from langchain_together import ChatTogether
import os

openai_chat_model = ChatTogether(
    model="openai/gpt-oss-20b",
    together_api_key=os.environ["TOGETHER_API_KEY"],
)

In [10]:
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser


class State(TypedDict):
    question: str
    context: list[Document]
    response: str


def retrieve(state: State) -> State:
    retrieved_docs = qdrant_retriever.invoke(state["question"])
    return {"context": retrieved_docs}


def generate(state: State) -> State:
    generator_chain = chat_prompt | openai_chat_model | StrOutputParser()
    response = generator_chain.invoke(
        {"query": state["question"], "context": state["context"]}
    )
    return {"response": response}


graph_builder = StateGraph(State)
graph_builder = graph_builder.add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
rag_graph = graph_builder.compile()

In [11]:
result = rag_graph.invoke({"question": "What is the maximum loan amount?"})
from IPython.display import Markdown, display

# Assuming the response is under the 'response' key
display(Markdown(result.get("response", "")))



The highest combined loan limit shown in the document is **$138,500** for graduate and professional students.